# CovIntervene frozen P0 backbone screening

This notebook executes all **64 series × 4 mechanisms × 3 generator seeds** for one frozen backbone. It writes resumable unit artifacts to Google Drive and is structurally unable to compute the continuation gate. Run it once for `chronos_2` and once for `timesfm_3`. Start with T4; switch to A100 only if runtime is impractical.

In [ ]:
BACKBONE = 'chronos_2'  # change to 'timesfm_3' for the second full run
assert BACKBONE in {'chronos_2', 'timesfm_3'}
print('Selected frozen backbone:', BACKBONE)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess

REPO = Path('/content/tsfm-covariate-faithfulness')
REPO_URL = 'https://github.com/FlyMe2star/tsfm-covariate-faithfulness.git'
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'main', REPO_URL, str(REPO)], check=True)
print('Repository commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
import os
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements/colab-base.txt')], check=True)
model_requirements = REPO / ('requirements/chronos2.txt' if BACKBONE == 'chronos_2' else 'requirements/timesfm3.txt')
if BACKBONE == 'timesfm_3':
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'timesfm'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(model_requirements)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
src_path = str(REPO / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
if BACKBONE == 'timesfm_3':
    from timesfm3 import ModelConfig, TimesFM3Evaluator
    print('Verified pinned TimesFM-3 source interface:', ModelConfig, TimesFM3Evaluator)
import covfaith
print('Dependencies installed for', BACKBONE)
print('covfaith imported from:', covfaith.__file__)

In [ ]:
import json
import torch

from covfaith.config import load_yaml, verify_config_lock
from covfaith.p0 import scientific_code_hash

EXPECTED_CONFIG_HASH = '56cd2ff3a3e56155074a47abb02a859be5e107a6fb59679f9f3d3981110db0dd'
EXPECTED_SCIENTIFIC_CODE_SHA256 = 'ee75398fe2b7c93ac2e2fa83b0abeeb76f346a8953bfc7c91e1a1cfbd6b5bcde'
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, then restart.'
config_path = REPO / 'configs/p0/covintervene_p0.yaml'
lock_path = REPO / 'configs/p0/covintervene_p0.lock.json'
config_hash = verify_config_lock(config_path, lock_path)
code_hash = scientific_code_hash(REPO)
assert config_hash == EXPECTED_CONFIG_HASH
assert code_hash == EXPECTED_SCIENTIFIC_CODE_SHA256
test_env = os.environ.copy()
test_env['PYTHONPATH'] = src_path
subprocess.run([sys.executable, '-m', 'pytest', '-q', str(REPO / 'tests')], cwd=REPO, env=test_env, check=True)
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
print('Frozen config hash:', config_hash[:12])
print('Frozen scientific code hash:', code_hash[:12])

In [ ]:
from covfaith.adapters import Chronos2Adapter, TimesFM3Adapter

config = load_yaml(config_path)
model = next(item for item in config['models'] if item['id'] == BACKBONE)
if BACKBONE == 'chronos_2':
    adapter = Chronos2Adapter.from_pretrained(
        model['checkpoint'], model['revision'], device='cuda', batch_size=128
    )
else:
    adapter = TimesFM3Adapter.from_pretrained(
        model['checkpoint'], model['revision'], device='cuda', per_core_batch_size=16
    )
print('Frozen checkpoint loaded:', model['checkpoint'], model['revision'])

In [ ]:
from covfaith.p0 import run_backbone_units

OUTPUT_ROOT = Path('/content/drive/MyDrive/tsfm-covariate-faithfulness/p0_full_v1')
report = run_backbone_units(REPO, adapter, OUTPUT_ROOT)
assert report['mode'] == 'full_screening'
assert report['series_per_mechanism_per_seed'] == 64
assert report['completed_unit_count'] == 12
assert report['scientific_gate_computed'] is False
assert report['config_hash'] == EXPECTED_CONFIG_HASH
assert report['scientific_code_sha256'] == EXPECTED_SCIENTIFIC_CODE_SHA256
print(json.dumps(report, indent=2, ensure_ascii=False))
print('Full screening units:', OUTPUT_ROOT)
print('No continuation gate was computed in this notebook.')